In [1]:
from typing import TypedDict,Annotated
from operator import add
from langgraph.graph import StateGraph,END
from langgraph.types import Command,interrupt
from langgraph.checkpoint.memory import MemorySaver

class BaiscChatState(TypedDict):
    text:Annotated[str, add]

memory = MemorySaver()


def node_a(state:BaiscChatState):
    print("node_a")
    return Command(
        goto="node_b",
        update={ "text":state["text"]+'a'  }
        ) 

def node_b(state:BaiscChatState):
    print("node_b")
    human_response=interrupt("do you want to go on C or D node?")
    print("Human Review Values: ", human_response)
    if human_response=="C":
        return Command(
            goto="node_c",
            update={"text":state["text"]+'b' }
        )
    
    elif human_response=="D":
        return Command(
        goto="node_d",
        update={"text":state["text"]+'b' }
        )
    

def node_c(state:BaiscChatState):
    print("node_c")
    return Command(
        goto="node_d",
        update={
            "text":state["text"]+'c'
        
    }) 

def node_d(state:BaiscChatState):
    print("node_d")
    return Command(
        goto=END,
        update={
            "text":state["text"]+'d'
        
    }) 



In [2]:
graph = StateGraph(BaiscChatState)

graph.add_node("node_a", node_a)
graph.add_node("node_b", node_b)
graph.add_node("node_c", node_c)
graph.add_node("node_d", node_d)

graph.set_entry_point("node_a")
app = graph.compile( checkpointer=memory)

config = {"configurable": {"thread_id": "1"}}
initialstate = {"text":""}
result = app.invoke(initialstate,config=config,stream_mode="updates")
# print(app.get_state(config=config).next)


node_a
node_b


In [ ]:
second_result=app.invoke(Command(resume="D"),config=config,stream_mode="updates")
second_result